# Yêu cầu 3: Model Versioning & Registry
- Đăng ký model tốt nhất vào MLflow Model Registry
- Gán stage: Staging → Production
- So sánh 2 version: model nào tốt hơn? Vì sao?

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri('file:../2_ModelTraining/mlruns')
client = MlflowClient()

## 1. Lấy 2 run đã train, chọn run tốt nhất theo F1

In [ ]:
exp = client.get_experiment_by_name('spam_classification')
runs = client.search_runs(exp.experiment_id, order_by=['metrics.test_f1 DESC'])
for r in runs:
    print(r.data.tags.get('mlflow.runName'), '| test_f1 =', r.data.metrics.get('test_f1'))
best_run = runs[0]
print('\nBest run:', best_run.data.tags.get('mlflow.runName'))

## 2. Đăng ký 2 version model vào Registry
- Version 1: Naive Bayes
- Version 2: XGBoost

In [ ]:
MODEL_NAME = 'spam_classifier'

for r in runs:
    model_uri = f'runs:/{r.info.run_id}/model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)
    print('Registered version', mv.version, 'from run', r.data.tags.get('mlflow.runName'))

## 3. Gán stage: Staging và Production

In [ ]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
versions = sorted(versions, key=lambda v: int(v.version))

client.transition_model_version_stage(MODEL_NAME, versions[0].version, 'Staging')
client.transition_model_version_stage(MODEL_NAME, versions[1].version, 'Production')

for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(f'Version {v.version}: stage = {v.current_stage}')

## 4. So sánh 2 version

In [ ]:
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    run = client.get_run(v.run_id)
    print(f"Version {v.version} ({run.data.tags.get('mlflow.runName')}): "
          f"P={run.data.metrics.get('test_precision'):.4f} "
          f"R={run.data.metrics.get('test_recall'):.4f} "
          f"F1={run.data.metrics.get('test_f1'):.4f}")

## 5. Giải thích ngắn (3-5 dòng)

**Model tốt hơn:** XGBoost (Production).

**Vì sao:**
1. XGBoost có F1 cao hơn Naive Bayes trên tập test → cân bằng tốt giữa Precision và Recall.
2. XGBoost xử lý được tương tác phi tuyến giữa các đặc trưng TF-IDF, trong khi Naive Bayes giả định độc lập giữa các từ.
3. XGBoost cũng hỗ trợ regularization, ít overfit hơn trên dataset có nhiều đặc trưng thưa.
4. Naive Bayes được giữ làm baseline ở stage Staging để fallback nếu XGBoost gặp sự cố.